In [1]:
"""
STEP 1 — GENERATION (needs the "kokoro" conda env).

For every story × version, synthesizes the audio and saves the complete result
(audio + every token's duration / word / alteration) as
    outputs/story<id>__<version>.html
That file is the saved data; build_experiment.py reads it later (no Kokoro needed).

To add a story: add it to STORIES and run again. With SKIP_EXISTING = True,
versions that already exist in outputs/ are not generated again.
"""
import time
import torch
import soundfile as sf
from pathlib import Path
from functions import (
    text_to_ipa_chunks, manipulate, check_ipa,
    synth_aligned, audiovisualize_interactive,
)

VOICES_DIR = Path("custom_voices")
OUT_DIR = Path("output_local"); OUT_DIR.mkdir(exist_ok=True)
SKIP_EXISTING = True      # don't regenerate versions that are already in outputs/

voice     = torch.load(VOICES_DIR / "nicola_80.pt",  weights_only=True)   # the voice you hear
dur_voice = torch.load(VOICES_DIR / "nicola_100.pt", weights_only=True)   # reference timing

# ---------------------------------------------------------------------------
# stories — id: text. The id is what participants choose and what is stored.
# ---------------------------------------------------------------------------

with open("story1.txt", "r", encoding="utf-8") as f:
    story1 = f.read()
story1 = story1[:200] 
STORIES = {
    "1": story1,
}

# ---------------------------------------------------------------------------
# error rule sets — add a dict here for a new kind of error
# ---------------------------------------------------------------------------
RULESETS = {
    "consistent": [
        {"kind": "substitute_prob", "old": "d", "choices": {"ʈ": 100}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "k", "choices": {"χ": 100}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "e", "choices": {"ø": 100}, "same_in_word": True},
    ],
    "less_consistent": [
        {"kind": "substitute_prob", "old": "d", "choices": {"ʈ": 33, "θ": 33, "dʲ": 33}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "k", "choices": {"x": 33, "χ": 33, "kʲ": 33}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "e", "choices": {"ø": 50, "e": 50}, "same_in_word": False},
        {"kind": "substitute_prob", "old": "o", "choices": {"ə": 50, "o": 50}, "same_in_word": False},
    ],
}

# timing variants; comment one out to skip it
TIMINGS = {
    "native":   lambda ipa, origins: dict(dur_source="original", original_chunks=ipa, origins=origins),
    # "accented": lambda ipa, origins: dict(dur_source="manipulated"),
}


def render(name, text, chunks, speed, masks=None, rules=None, stats=None, **timing):
    if SKIP_EXISTING and (OUT_DIR / f"{name}.html").exists():
        print(f"  {name}: already in outputs/, skipped")
        return
    t = time.time()
    audio, segments, seg_words, seg_altered = synth_aligned(
        chunks, voice, speed, masks=masks, dur_voice=dur_voice, **timing)
    audiovisualize_interactive(
        audio, segments, seg_words=seg_words, text=text, seg_altered=seg_altered,
        title=name, rules=rules, stats=stats, out_html=f"{name}.html")
    sf.write(OUT_DIR / f"{name}.wav", audio, 24000)
    print(f"  {name}: {len(audio)/24000:.1f}s audio in {time.time()-t:.1f}s")

for speed in [0.8, 0.9, 0.85]:

    for sid, text in STORIES.items():
        print(f"story {sid}")
        ipa_chunks = text_to_ipa_chunks(text)
        check_ipa(ipa_chunks)
        render(f"test-{speed}-story{sid}__clean", text, ipa_chunks, speed, dur_source="manipulated")
        for rname, rules in RULESETS.items():
            manip_chunks, masks, stats, origins = manipulate(
                ipa_chunks, rules, return_mask=True, return_stats=True, return_origin=True, seed=42)
            for tname, timing in TIMINGS.items():
                render(f"test-{speed}-story{sid}__{rname}_dur_forced_{tname}", text, manip_chunks, speed, masks, rules, stats,
                    **timing(ipa_chunks, origins))

print("done — saved pages are in ./outputs/. Next: python build_experiment.py")

c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


story 1
[chunk 0] 'esplorˈare il fˈondo dˌeʎɪ oʧˈɛanɪ e ʊnimprˈeza inkredibilmˈente diffˈiʧile, mˈolto pjʊ komplˈeSa dɪ ʊn vjˈaʤːo nˈello spˈaʦio.'
   model sees : 'esplorˈare il fˈondo dˌeʎɪ oʧˈɛanɪ e ʊnimprˈeza inkredibilmˈente diffˈiʧile, mˈolto pjʊ komplˈeSa dɪ ʊn vjˈaʤːo nˈello spˈaʦio.'
   tokens     : [47, 61, 58, 54, 57, 60, 156, 43, 60, 47, 16, 51, 54, 16, 48, 156, 57, 56, 46, 57, 16, 46, 157, 47, 143, 102, 16, 57, 133, 156, 86, 43, 56, 102, 16, 47, 16, 135, 56, 51, 55, 58, 60, 156, 47, 68, 43, 16, 51, 56, 53, 60, 47, 46, 51, 44, 51, 54, 55, 156, 47, 56, 62, 47, 16, 46, 51, 48, 48, 156, 51, 133, 51, 54, 47, 3, 16, 55, 156, 57, 54, 62, 57, 16, 58, 52, 135, 16, 53, 57, 55, 58, 54, 156, 47, 35, 43, 16, 46, 102, 16, 135, 56, 16, 64, 52, 156, 43, 82, 158, 57, 16, 56, 156, 47, 54, 54, 57, 16, 61, 58, 156, 43, 20, 51, 57, 4]
   ok

[chunk 1] 'kwˈando ʊn astronˈaʊta sɪ trˈɔva nˈello spˈaʦio estˈɛrno, la differˈɛnʦa tra lˈaria dˈɛn'
   model sees : 'kwˈando ʊn astronˈaʊta sɪ trˈɔva nˈe